# Logistic Regression: 90-Day Churn

Train a baseline logistic regression model from `data/features_and_label.parquet`, where `is_churn` is the target. The final 20% of customer-months by time is held out so evaluation reflects predicting later periods. Customer identifiers and the month itself are excluded from model inputs.


## 1. Load and inspect the feature table

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    roc_auc_score,
    average_precision_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
import matplotlib.pyplot as plt

DATA_PATH = Path('../data/features_and_label.parquet')
df = pd.read_parquet(DATA_PATH)

TARGET = 'is_churn'
required_columns = {'CustomerMonth', TARGET}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

# CustomerMonth may be stored as a pandas Period (for example, Period[M]) in parquet.
# Convert periods to their month-start timestamps; parse other representations normally.
customer_month = df['CustomerMonth']
if isinstance(customer_month.dtype, pd.PeriodDtype):
    df['CustomerMonth'] = customer_month.dt.to_timestamp()
else:
    df['CustomerMonth'] = pd.to_datetime(customer_month)
df = df.dropna(subset=[TARGET, 'CustomerMonth']).copy()
df[TARGET] = df[TARGET].astype(int)

print(f'Rows: {len(df):,} | churn rate: {df[TARGET].mean():.2%}')
print(f"Date range: {df['CustomerMonth'].min()} through {df['CustomerMonth'].max()}")
display(df.head())


Rows: 20,880 | churn rate: 38.99%
Date range: 2009-12-01 00:00:00 through 2011-09-01 00:00:00


,Customer ID,CustomerMonth,purchase_frequency,total_spend,avg_order_value,unique_items,days_since_last_purchase,is_first_month,is_churn
0,12346,2010-03-01,1,27.05,27.05,5,NaN,True,1
1,12346,2010-06-01,1,142.31,142.31,19,118.0,False,1
2,12346,2011-01-01,1,149.76,149.76,1,203.0,False,1
3,12347,2010-10-01,1,611.53,611.53,40,NaN,True,0
4,12347,2010-12-01,1,711.79,711.79,31,37.0,False,0


## 2. Create chronological train and test sets

Rows before the cutoff train the model; the latest 20% of distinct months are reserved for evaluation. This keeps later observations out of training.

In [2]:
months = np.sort(df['CustomerMonth'].dropna().unique())
if len(months) < 2:
    raise ValueError('At least two distinct customer months are needed for a chronological split.')

cutoff_index = max(1, int(np.floor(len(months) * 0.8)))
cutoff_index = min(cutoff_index, len(months) - 1)
cutoff_month = months[cutoff_index]

train_df = df[df['CustomerMonth'] < cutoff_month].copy()
test_df = df[df['CustomerMonth'] >= cutoff_month].copy()

# Customer ID is an identifier, CustomerMonth defines the split; neither is a predictor.
feature_cols = [
    column for column in df.columns
    if column not in {'Customer ID', 'CustomerMonth', TARGET}
]
if not feature_cols:
    raise ValueError('No predictor columns found after excluding ID, date, and target.')

X_train, y_train = train_df[feature_cols], train_df[TARGET]
X_test, y_test = test_df[feature_cols], test_df[TARGET]

if y_train.nunique() < 2:
    raise ValueError('Training split contains only one label class; logistic regression needs both classes.')
if y_test.nunique() < 2:
    print('Warning: test split contains one class; ROC AUC will not be available.')

print(f'Cutoff month: {pd.Timestamp(cutoff_month):%Y-%m}')
print(f'Train: {len(train_df):,} rows, {train_df.CustomerMonth.nunique()} months, churn={y_train.mean():.2%}')
print(f'Test:  {len(test_df):,} rows, {test_df.CustomerMonth.nunique()} months, churn={y_test.mean():.2%}')
print('Predictors:', feature_cols)


Cutoff month: 2011-05
Train: 16,686 rows, 17 months, churn=39.94%
Test:  4,194 rows, 5 months, churn=35.24%
Predictors: ['purchase_frequency', 'total_spend', 'avg_order_value', 'unique_items', 'days_since_last_purchase', 'is_first_month']


## 3. Train and Evaluate logistic regression

Numeric feature gaps are median-imputed and features are standardized. Balanced class weights account for churn class imbalance. The preprocessing is fit on training data only.


In [3]:
# Logistic regression benefits from standardized inputs.
numeric_preprocessor = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
)
logistic_model = make_pipeline(
    make_column_transformer(
        (numeric_preprocessor, feature_cols),
        remainder='drop',
        verbose_feature_names_out=False,
    ),
    LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=42,
    ),
)
logistic_model.fit(X_train, y_train)
print('Logistic regression fitted.')


Logistic regression fitted.


In [4]:
logistic_y_pred = logistic_model.predict(X_test)
logistic_y_prob = logistic_model.predict_proba(X_test)[:, 1]

logistic_regression_results = pd.DataFrame([{
    'Accuracy': accuracy_score(y_test, logistic_y_pred),
    'Precision': precision_score(y_test, logistic_y_pred, zero_division=0),
    'Recall': recall_score(y_test, logistic_y_pred, zero_division=0),
    'F1': f1_score(y_test, logistic_y_pred, zero_division=0),
    'ROC AUC': roc_auc_score(y_test, logistic_y_prob) if y_test.nunique() == 2 else np.nan,
    'PR AUC': average_precision_score(y_test, logistic_y_prob) if y_test.nunique() == 2 else np.nan,
}], index=['Logistic regression'])
display(logistic_regression_results)


,Accuracy,Precision,Recall,F1,ROC AUC,PR AUC
Logistic regression,0.713877,0.586228,0.639378,0.61165,0.765619,0.625692


## 4. Train random forest

Median imputation handles first-month recency gaps. Class weighting gives the less frequent churn class more influence during training.


In [5]:
# Trees do not need feature scaling.
forest_model = make_pipeline(
    SimpleImputer(strategy='median'),
    RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    ),
)
forest_model.fit(X_train, y_train)
print('Random forest fitted.')


Random forest fitted.


In [6]:
forest_y_pred = forest_model.predict(X_test)
forest_y_prob = forest_model.predict_proba(X_test)[:, 1]

random_forest_results = pd.DataFrame([{
    'Accuracy': accuracy_score(y_test, forest_y_pred),
    'Precision': precision_score(y_test, forest_y_pred, zero_division=0),
    'Recall': recall_score(y_test, forest_y_pred, zero_division=0),
    'F1': f1_score(y_test, forest_y_pred, zero_division=0),
    'ROC AUC': roc_auc_score(y_test, forest_y_prob) if y_test.nunique() == 2 else np.nan,
    'PR AUC': average_precision_score(y_test, forest_y_prob) if y_test.nunique() == 2 else np.nan,
}], index=['Random forest'])
display(random_forest_results)


,Accuracy,Precision,Recall,F1,ROC AUC,PR AUC
Random forest,0.675966,0.53548,0.607578,0.569255,0.719131,0.544215


## 5. Train XGBoost

Median imputation keeps the feature handling consistent with the other models. `scale_pos_weight` adjusts for class imbalance using the training labels only.


In [7]:
xgb_model = make_pipeline(
    SimpleImputer(strategy='median'),
    XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1,
    ),
)
xgb_model.fit(X_train, y_train)
print('XGBoost fitted.')


XGBoost fitted.


In [8]:
xgb_y_pred = xgb_model.predict(X_test)
xgb_y_prob = xgb_model.predict_proba(X_test)[:, 1]

xgboost_results = pd.DataFrame([{
    'Accuracy': accuracy_score(y_test, xgb_y_pred),
    'Precision': precision_score(y_test, xgb_y_pred, zero_division=0),
    'Recall': recall_score(y_test, xgb_y_pred, zero_division=0),
    'F1': f1_score(y_test, xgb_y_pred, zero_division=0),
    'ROC AUC': roc_auc_score(y_test, xgb_y_prob) if y_test.nunique() == 2 else np.nan,
    'PR AUC': average_precision_score(y_test, xgb_y_prob) if y_test.nunique() == 2 else np.nan,
}], index=['XGBoost'])
display(xgboost_results)


,Accuracy,Precision,Recall,F1,ROC AUC,PR AUC
XGBoost,0.696948,0.554388,0.713802,0.624076,0.76795,0.622006


## 6. Compare models

All three models on the same test set. PR AUC is the area under the precision-recall curve; a model that guesses at random scores the test churn rate, so that rate is shown as the baseline.


In [9]:
model_comparison = pd.concat([
    logistic_regression_results,
    random_forest_results,
    xgboost_results,
])
print(f'PR AUC baseline (test churn rate): {y_test.mean():.3f}')
display(model_comparison.round(3))


PR AUC baseline (test churn rate): 0.352


,Accuracy,Precision,Recall,F1,ROC AUC,PR AUC
Logistic regression,0.714,0.586,0.639,0.612,0.766,0.626
Random forest,0.676,0.535,0.608,0.569,0.719,0.544
XGBoost,0.697,0.554,0.714,0.624,0.768,0.622


## Notes

- The latest 20% of distinct customer months are held out, and all three models use the same split and predictors.
- Customer identifiers and month are excluded from predictors. This evaluates later observations for customers who may also appear in training.
- Each estimator is fit in its own cell; run cells 1?6 in order.
